Instalaciones necesarias: Numpy, pettingzoo

In [ ]:
%pip install pettingzoo "stable-baselines3[extra]" supersuit torch matplotlib seaborn

In [ ]:
%pip install stable-baselines3[extra]

In [ ]:
%pip install supersuit

In [ ]:
%pip install numpy

Codigo

In [4]:
import os
import random
import numpy as np
import gymnasium
from gymnasium import spaces

# --- Importaciones de Stable-Baselines3 (Entrenamiento) ---
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.env_checker import check_env 

# Esta es la importacion para exportar los datos
from stable_baselines3.common.logger import configure
from stable_baselines3.common.monitor import Monitor


################################################################################
#                                                                              #
#                      PARTE 1: DEFINICIÓN DEL ENTORNO (GYMNASIUM)             #
#                                                                              #
#                                                                              #
################################################################################

# --- Definiciones de Acciones ---
ACCION_MOVER_N = 0
ACCION_MOVER_S = 1
ACCION_MOVER_E = 2
ACCION_MOVER_O = 3
ACCION_RECOLECTAR = 4
ACCION_COMER = 5
ACCION_DESCANSAR = 6
NUM_ACCIONES = 7

# --- Definiciones del Mapa ---
MAPA_SUELO = 0
MAPA_COMIDA = 1    # Tienda / Arbusto
MAPA_PELIGRO = 2   # Derrumbe / Animal
MAPA_DESCANSO = 3  # Casa / Vegetación

class AgenteInterno:
    def __init__(self, pos_x, pos_y):
        self.x = pos_x
        self.y = pos_y
        self.salud = 5
        self.hambre = 5
        self.energia = 5
        self.inventario_comida = 0
        self.esta_vivo = True

class EntornoSupervivenciaGym(gymnasium.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, ancho=10, alto=10, max_pasos=100):
        super().__init__()
        
        self.ancho = ancho
        self.alto = alto
        self.max_pasos_por_episodio = max_pasos
        
        self.action_space = spaces.Discrete(NUM_ACCIONES) 
        
        obs_low = np.array([0, 0, 0, 0, 0, 0, 0])
        obs_high = np.array([5, 5, 5, 1, self.ancho-1, self.alto-1, 3])
        self.observation_space = spaces.Box(low=obs_low, high=obs_high, dtype=np.int32)
        
        self.mapa_base = np.zeros((self.alto, self.ancho), dtype=int)
        self.pasos_actuales = 0
        self.agente = None

    def _generar_mapa(self):
        self.mapa_base.fill(MAPA_SUELO)
        for _ in range(int(self.ancho * self.alto * 0.1)):
            x, y = self._posicion_aleatoria_libre()
            self.mapa_base[y, x] = MAPA_COMIDA
        for _ in range(int(self.ancho * self.alto * 0.05)):
            x, y = self._posicion_aleatoria_libre()
            self.mapa_base[y, x] = MAPA_PELIGRO
        for _ in range(int(self.ancho * self.alto * 0.05)):
            x, y = self._posicion_aleatoria_libre()
            self.mapa_base[y, x] = MAPA_DESCANSO

    def _posicion_aleatoria_libre(self):
        while True:
            x = random.randint(0, self.ancho - 1)
            y = random.randint(0, self.ancho - 1)
            if self.mapa_base[y, x] == MAPA_SUELO:
                return x, y

    def _obtener_observacion(self):
        casilla_actual = self.mapa_base[self.agente.y, self.agente.x]
        return np.array([
            self.agente.salud,
            self.agente.hambre,
            self.agente.energia,
            self.agente.inventario_comida,
            self.agente.x,
            self.agente.y,
            casilla_actual
        ], dtype=np.int32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._generar_mapa()
        self.pasos_actuales = 0
        x, y = self._posicion_aleatoria_libre()
        self.agente = AgenteInterno(x, y)
        observacion = self._obtener_observacion()
        info = {}
        return observacion, info

    def step(self, accion):
        self.pasos_actuales += 1
        recompensa = -0.01
        
        if self.agente.energia == 0 and accion != ACCION_DESCANSAR:
            recompensa -= 0.1
        else:
            if accion != ACCION_DESCANSAR:
                self.agente.energia -= 1
            if accion == ACCION_MOVER_N and self.agente.y > 0: self.agente.y -= 1
            elif accion == ACCION_MOVER_S and self.agente.y < self.alto - 1: self.agente.y += 1
            elif accion == ACCION_MOVER_E and self.agente.x < self.ancho - 1: self.agente.x += 1
            elif accion == ACCION_MOVER_O and self.agente.x > 0: self.agente.x -= 1
            elif accion == ACCION_RECOLECTAR:
                if self.mapa_base[self.agente.y, self.agente.x] == MAPA_COMIDA:
                    if self.agente.inventario_comida == 0:
                        self.agente.inventario_comida = 1
                        recompensa += 0.1
                    else: recompensa -= 0.1
                else: recompensa -= 0.1
            elif accion == ACCION_COMER:
                if self.agente.inventario_comida == 1:
                    self.agente.inventario_comida = 0
                    self.agente.hambre = 5
                else: recompensa -= 0.1
            elif accion == ACCION_DESCANSAR:
                if self.mapa_base[self.agente.y, self.agente.x] == MAPA_DESCANSO:
                    self.agente.energia = 5
                else:
                    self.agente.energia = min(5, self.agente.energia + 3)

        if self.pasos_actuales % 5 == 0:
            self.agente.hambre = max(0, self.agente.hambre - 1)
        if self.agente.hambre == 0:
            self.agente.salud -= 1
            recompensa -= 0.5
        if self.mapa_base[self.agente.y, self.agente.x] == MAPA_PELIGRO:
            self.agente.salud -= 1
            recompensa -= 0.5
            
        terminado = False
        truncado = False
        if self.agente.salud <= 0:
            recompensa -= 0.5
            terminado = True
            self.agente.esta_vivo = False
        if self.pasos_actuales >= self.max_pasos_por_episodio:
            if self.agente.esta_vivo:
                recompensa += 1.0
            truncado = True
        
        observacion = self._obtener_observacion()
        info = {}
        return observacion, recompensa, terminado, truncado, info

    def render(self, mode="human"):
        if mode == "human":
            mapa_render = np.full((self.alto, self.ancho), " . ")
            mapa_render[self.mapa_base == MAPA_COMIDA] = " F "
            mapa_render[self.mapa_base == MAPA_PELIGRO] = " X "
            mapa_render[self.mapa_base == MAPA_DESCANSO] = " H "
            if self.agente.esta_vivo:
                mapa_render[self.agente.y, self.agente.x] = " A "
            else:
                mapa_render[self.agente.y, self.agente.x] = " † "
            print("-" * (self.ancho * 3 + 2))
            for fila in mapa_render:
                print(f"|{''.join(fila)}|")
            print("-" * (self.ancho * 3 + 2))
            ag = self.agente
            print(f"Agente: S:{ag.salud} H:{ag.hambre} E:{ag.energia} C:{ag.inventario_comida}")

    def close(self):
        pass


################################################################################
#                                                                              #
#             PARTE 2: LÓGICA DE ENTRENAMIENTO                                 #
#                                                                              #
################################################################################

# --- 1. Configuración de Directorios ---
LOG_DIR = "logs/" 
MODEL_DIR = "models/" 
MODEL_NAME = "ppo_supervivencia_1agente" 
TOTAL_TIMESTEPS = 100_000 #Cantidad de salgos que realiza

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# --- 2. Función para crear el entorno ---
def crear_entorno():
    env = EntornoSupervivenciaGym()
    env = Monitor(env)
    return env

# --- 3. Instanciar y "envolver" el entorno ---
print("Creando y envolviendo el entorno...")
env_base = crear_entorno()
env = DummyVecEnv([lambda: env_base])

log_path = os.path.join(LOG_DIR, MODEL_NAME)

print(f"Configurando Logger en: {log_path}")
new_logger = configure(log_path, ["stdout", "csv", "tensorboard"])


# --- 5. Definir el Modelo (PPO) ---
print("Configurando el modelo PPO...")
model = PPO(
    policy='MlpPolicy',
    env=env,
    n_steps=1024,        
    batch_size=64,
    n_epochs=10,
    gamma=0.99,          
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01

)

# Esto le dice al modelo que use nuestra configuración
model.set_logger(new_logger)


# --- 7. Entrenar el Modelo ---
if __name__ == "__main__":
    print(f"--- Iniciando entrenamiento PPO por {TOTAL_TIMESTEPS} timesteps ---")
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        
    )
    print("--- Entrenamiento finalizado ---")

    # --- 8. Guardar el Modelo ---
    model_path = os.path.join(MODEL_DIR, f"{MODEL_NAME}.zip")
    model.save(model_path)
    print(f"Modelo guardado exitosamente en: {model_path}")

    env.close()

Creando y envolviendo el entorno...
Configurando Logger en: logs/ppo_supervivencia_1agente
Logging to logs/ppo_supervivencia_1agente


Configurando el modelo PPO...
--- Iniciando entrenamiento PPO por 100000 timesteps ---
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 28       |
|    ep_rew_mean     | -4.78    |
| time/              |          |
|    fps             | 1458     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 1024     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 28.7        |
|    ep_rew_mean          | -4.71       |
| time/                   |             |
|    fps                  | 996         |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 2048        |
| train/                  |             |
|    approx_kl            | 0.012784297 |
|    clip_fraction        | 0.158       |
|    clip_range           | 0.2         |
|    entropy_loss         |